# 🧪 AWS Architecture Prompt Evaluation Lab

Este notebook te permite seleccionar un video, preparar su entorno de laboratorio (copiando imágenes, transcripciones y grafos previos), probar un nuevo prompt personalizado con la API de Gemini Vision, y comparar los resultados de rendimiento (F1-score de servicios y aristas) de forma interactiva y detallada.

### 1. Configuración de Rutas e Importaciones

In [2]:
import sys
import os
import json
import csv
import shutil
from pathlib import Path
import networkx as nx
import pandas as pd

# Configurar la ruta raíz del proyecto
PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
sys.path.append(str(PROJECT_ROOT))

# Importar módulos del proyecto para construir y evaluar grafos
from scripts.graph_builder import create_graph_from_cloudscape_json, export_graphml
from scripts.evaluate_graphs import evaluate_pair, load_services_catalog

print("✓ Módulos del proyecto cargados con éxito.")

✓ Módulos del proyecto cargados con éxito.


### 2. Selección del Video y Preparación del Workspace
Elige el `VIDEO_ID` (por ejemplo, `BgT_bDAejSQ`) que desees analizar. El script copiará todos los archivos relacionados a la carpeta de trabajo del laboratorio `whiteboard_selection_lab/lab_workspace/<VIDEO_ID>`.

In [3]:
# ID del video a probar (ejemplos: BgT_bDAejSQ, 3yJZ6rPoZfg, BPvr0qWpJlA, etc.)
VIDEO_ID = "-kA0ahrhX3I"

# Definir el directorio del laboratorio para este video
LAB_WORKSPACE = PROJECT_ROOT / "whiteboard_selection_lab" / "lab_workspace" / VIDEO_ID
LAB_WORKSPACE.mkdir(parents=True, exist_ok=True)

# Rutas de origen
GT_GRAPH_PATH = PROJECT_ROOT / "data" / "cloudscape_gt" / f"{VIDEO_ID}.graphml"
STD_GRAPH_PATH = PROJECT_ROOT / "data" / "graphs" / f"{VIDEO_ID}.graphml"
PARSI_GRAPH_PATH = PROJECT_ROOT / "data" / "graphs_parsimonious" / f"{VIDEO_ID}.graphml"
GOOD_WHITEBOARD = PROJECT_ROOT / "data" / "good_whiteboard" / f"{VIDEO_ID}.jpg"
TRANSCRIPT_PATH = PROJECT_ROOT / "data" / "raw" / f"{VIDEO_ID}_transcript.json"

# Rutas de destino
ws_gt_graph = LAB_WORKSPACE / "gt_graph.graphml"
ws_std_graph = LAB_WORKSPACE / "std_graph.graphml"
ws_parsi_graph = LAB_WORKSPACE / "parsi_graph.graphml"
ws_whiteboard = LAB_WORKSPACE / "whiteboard.jpg"
ws_transcript = LAB_WORKSPACE / "transcript.json"

print(f"🚀 Preparando entorno de pruebas para el video: {VIDEO_ID} en {LAB_WORKSPACE}...")

if GOOD_WHITEBOARD.exists():
    shutil.copy(GOOD_WHITEBOARD, ws_whiteboard)
    print(f"  ✓ Pizarra buena copiada.")
else:
    print(f"  ✗ Error: No se encontró la pizarra aprobada en {GOOD_WHITEBOARD}")

if TRANSCRIPT_PATH.exists():
    shutil.copy(TRANSCRIPT_PATH, ws_transcript)
    print(f"  ✓ Transcripción JSON copiada.")
else:
    print(f"  ✗ Error: No se encontró la transcripción en {TRANSCRIPT_PATH}")

if GT_GRAPH_PATH.exists():
    shutil.copy(GT_GRAPH_PATH, ws_gt_graph)
    print(f"  ✓ Grafo Ground Truth copiado.")
else:
    print(f"  ✗ Error: Grafo Ground Truth no encontrado en {GT_GRAPH_PATH}")

if STD_GRAPH_PATH.exists():
    shutil.copy(STD_GRAPH_PATH, ws_std_graph)
    print(f"  ✓ Grafo Estándar original copiado.")
else:
    print(f"  - Nota: Grafo Estándar original no encontrado.")

if PARSI_GRAPH_PATH.exists():
    shutil.copy(PARSI_GRAPH_PATH, ws_parsi_graph)
    print(f"  ✓ Grafo Parsimonioso original copiado.")
else:
    print(f"  - Nota: Grafo Parsimonioso original no encontrado.")

print("\n🎉 Entorno preparado con éxito. Listo para experimentación.")

🚀 Preparando entorno de pruebas para el video: -kA0ahrhX3I en /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/-kA0ahrhX3I...
  ✓ Pizarra buena copiada.
  ✓ Transcripción JSON copiada.
  ✓ Grafo Ground Truth copiado.
  ✓ Grafo Estándar original copiado.
  ✓ Grafo Parsimonioso original copiado.

🎉 Entorno preparado con éxito. Listo para experimentación.


### 3. Definición del Prompt a Probar
Edita el contenido de `CUSTOM_SYSTEM_PROMPT` para experimentar con nuevas reglas o directivas. Las variables `<USER_ACTORS_PLACEHOLDER>` y `<AWS_SERVICES_PLACEHOLDER>` se reemplazarán automáticamente con los catálogos correspondientes.

In [4]:
# =====================================================================
# 2. PROMPT PARSIMONIOUS v2
# =====================================================================
CUSTOM_SYSTEM_PROMPT = """You are an expert AWS Solutions Architect. You are analyzing a whiteboard screenshot from an AWS "This is My Architecture" YouTube video, along with the full transcript of the video.

Your task is to extract the cloud architecture shown, encoding it using the Cloudscape dataset schema (FAST25 paper by Satija et al.) in a way that matches the style and level of detail of the manual ground truth dataset as closely as possible.

## RULES:
1. Use SHORT AWS service names for `service` field: e.g. "S3", "Lambda", "EC2", "DynamoDB", "EKS", "CloudFront", etc.

2. USER ACTOR NORMALIZATION: Only add User nodes that are EXPLICITLY shown as icons on the whiteboard OR mentioned as main actors in the transcript. Choose from this list: <USER_ACTORS_PLACEHOLDER>. To match Ground Truth style:
   - Map end-users accessing via browsers to "UserConsumerWeb", and app users to "UserConsumerMobile". Do NOT combine them into "UserConsumerWebMobile" unless a single physical box on the board is explicitly labeled for both.
   - Prefer "UserCompanyAgent" for internal operations teams, customer support agents, or backend system operators.
   - Use "UserCompanyDeveloper" for security engineers, data engineers, DevOps, or any internal technical staff who manage, query, or develop systems.
   - Use "UserCompanyAnalyst" ONLY when the persona is explicitly a data scientist or business analyst running reports.

3. Map rendering engine clusters/instances running on EC2 directly to service "EC2", putting "Rendering Engines" or "ASG" in the name or notes field.

4. NON-CLOUD & ON-PREMISE NORMALIZATION:
   - Do NOT use "ThirdParty" for internal microservices. Map them to the underlying AWS compute/storage service they run on (e.g. "EKS", "Lambda").
   - Map on-premises servers, local databases, legacy infrastructure, and third-party databases (MySQL, PostgreSQL, etc.) to "ThirdParty", unless a dedicated data center icon is explicitly drawn (in which case use "OnPremDC").

5. NODE MULTIPLICITY & ICON COUNTING:
   - The number of nodes must match the number of PHYSICAL ICONS (boxes, circles, labeled regions) drawn on the whiteboard.
   - If multiple identical service icons appear (e.g., two Macie icons in different AWS Account boxes), create a SEPARATE node for EACH physical icon — they represent distinct instances.
   - EXCEPTION: If the transcript explicitly says multiple visual icons represent a SINGLE shared resource, consolidate them into one node.
   - NO TRANSIENT ARTIFACTS: Do NOT create nodes for transient artifacts (e.g., "AMI", "Container Image"). Represent these actions as descriptions on edges.
   - TEXT-ONLY SERVICES: If a custom box merely lists services as text inside it (e.g., "ELB EC2 ASG") WITHOUT drawing individual AWS icons, DO NOT extract them as separate nodes.

6. Edges must have: flow_id (integer), seq (string), type ("data" or "meta"). Default to "data" for all edges. Only use "meta" for query-style edges where a service queries/reads metadata from a data store without transforming or moving the data (e.g., Athena querying S3, QuickSight reading from S3).

7. EDGE DIRECTIONALITY — FOLLOW THE ARROWS ON THE WHITEBOARD:
   - FOLLOW the physical arrow directions drawn on the whiteboard as closely as possible.
   - For data pipelines: use the direction of DATA FLOW (who produces data → who consumes it). Example: Macie writes findings → S3 bucket, so edge is Macie → S3.
   - For query/read operations: The querying service initiates the read. Model this as: DataStore → QueryService (data flows FROM the store TO the reader). Example: S3 → Athena means Athena queries S3 findings.
   - Do NOT add return/response paths or API acknowledgments UNLESS a SEPARATE, DISTINCT arrow is physically drawn on the whiteboard for the return path.
   - READ/WRITE AS DISTINCT FLOWS: If the whiteboard or transcript describes both writing TO and reading FROM a data store as separate operations, create BOTH edges (e.g., Macie → S3 for writes AND S3 → Athena for reads). These are NOT "return paths" — they are distinct data operations.

8. Minimize the number of flows. Group related sequential interactions into a single flow. A typical architecture should have 2-5 flows.

9. The `notes` field for nodes should capture context from the transcript: how the service is used.

10. WHITEBOARD IMAGE IS THE PRIMARY STRUCTURE GUIDE: The physical whiteboard image (icons and drawn arrows) is the primary source of truth for the structure of the graph. Do NOT add extra nodes or complex orchestration paths that are not represented by icons or arrows on the whiteboard.

## PARSIMONY PRINCIPLE:
Prefer FEWER nodes and edges over more. If you are unsure whether a service or connection exists in the architecture, DO NOT include it. It is better to miss a real service than to hallucinate a fake one.

## VALID SERVICE NAMES:
You MUST only use names from this list of canonical services when defining the `service` field in the nodes list (do not invent names or use raw abbreviations unless listed here):
<AWS_SERVICES_PLACEHOLDER>

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "step_by_step_reasoning": "First, count the physical icons on the whiteboard. Then trace every arrow. For each arrow, note: which icon it starts from, which icon it points to, and what kind of interaction it represents (data write, data read/query, trigger, etc.)...",
  "graph": {
    "name": "<title of the architecture>",
    "link": "<youtube URL if known, else empty string>",
    "categories": "<comma-separated from: data_ingestion, interactive, compute_intensive, control, other>",
    "graph_usable": true,
    "notes": "<distilled context>"
  },
  "nodes": [
    {"id": "0", "service": "...", "name": "", "notes": "..."}
  ],
  "edges": [
    {"source": "0", "target": "1", "flow_id": 0, "seq": "0", "type": "data", "notes": ""}
  ]
}
"""

print("✓ Prompt Parsimonious v2 definido.")

✓ Prompt Parsimonious v2 definido.


### 4. Ejecución del Experimento con Gemini API
Llamamos a la API de Gemini enviando el frame de pizarra copiado en el paso 2 junto con la transcripción completa, bajo el nuevo prompt.

In [5]:
import base64
from google import genai
from config.settings import GEMINI_API_KEY, GEMINI_MODEL

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# 1. Cargar y codificar imagen de pizarra local en base64
image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

# 2. Cargar transcripción de texto
with open(ws_transcript, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# 3. Cargar y formatear el catálogo oficial de servicios
services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

formatted_prompt = CUSTOM_SYSTEM_PROMPT.replace(
    "<USER_ACTORS_PLACEHOLDER>", ", ".join(sorted(user_actors_list))
).replace(
    "<AWS_SERVICES_PLACEHOLDER>", ", ".join(sorted(aws_services_list))
)

full_prompt = f"{formatted_prompt}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

print(f"📡 Enviando petición a Gemini utilizando el modelo: {GEMINI_MODEL}...")
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

raw_text = response.text.strip()
# Limpieza de posibles fences markdown
if raw_text.startswith("```"):
    raw_text = raw_text.split("\n", 1)[1]
    if "```" in raw_text:
        raw_text = raw_text[: raw_text.rfind("```")]
    raw_text = raw_text.strip()

test_result = json.loads(raw_text)

# Guardar resultado del experimento localmente
ws_test_json = LAB_WORKSPACE / "test_analysis.json"
ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"✅ Análisis de Gemini completado. Guardado en: {ws_test_json}")

📡 Enviando petición a Gemini utilizando el modelo: gemini-3.5-flash...


JSONDecodeError: Expecting ',' delimiter: line 119 column 4 (char 3858)

### 5. Compilación del Grafo de Prueba
Construimos el objeto de Grafo (NetworkX) a partir del JSON devuelto en la ejecución experimental.

In [4]:
test_graph = create_graph_from_cloudscape_json(test_result, video_id=VIDEO_ID)
test_graph_path = LAB_WORKSPACE / "test_graph.graphml"
nx.write_graphml(test_graph, str(test_graph_path))
print(f"✓ Grafo experimental exportado correctamente a: {test_graph_path}")

✓ Graph created: 11 nodes, 4 edges

✓ Grafo experimental exportado correctamente a: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/-kA0ahrhX3I/test_graph.graphml


### 6. Evaluación de Rendimiento y Comparación Detallada
Evaluamos la precisión, recall y F1-score (tanto a nivel de Nodos/Servicios como de Aristas/Conexiones) del nuevo enfoque y lo comparamos lado a lado con las ejecuciones anteriores.

In [5]:
# Cargar catálogo en dict para evaluate_pair
catalog_dict = {}
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        if name:
            catalog_dict[name] = {"capability": row.get("capability", "").strip()}

# Cargar grafos necesarios para comparación
gt_g = nx.read_graphml(ws_gt_graph)
test_g = nx.read_graphml(test_graph_path)

# Evaluar nuestro experimento (Nueva Prueba)
test_eval = evaluate_pair(test_g, gt_g, VIDEO_ID, catalog_dict)

# Funciones auxiliares para cargar y evaluar ejecuciones previas si existen
def evaluar_grafo_previo(path):
    if path.exists():
        try:
            g = nx.read_graphml(path)
            return evaluate_pair(g, gt_g, VIDEO_ID, catalog_dict)
        except Exception as e:
            return {"error": str(e)}
    return None

std_eval = evaluar_grafo_previo(ws_std_graph)
parsi_eval = evaluar_grafo_previo(ws_parsi_graph)

# Generar DataFrame comparativo
filas_comparativa = []

def agregar_fila(label, key, es_porcentaje=True):
    fila = {"Métrica": label}
    # Ground Truth
    if key == "gen_nodes":
        fila["Ground Truth"] = gt_g.number_of_nodes()
    elif key == "gen_edges":
        fila["Ground Truth"] = gt_g.number_of_edges()
    else:
        fila["Ground Truth"] = "—"
        
    # Standard
    if std_eval and key in std_eval:
        val = std_eval[key]
        fila["Standard Original"] = f"{val:.1%}" if es_porcentaje else val
    else:
        fila["Standard Original"] = "N/A"
        
    # Parsimonious
    if parsi_eval and key in parsi_eval:
        val = parsi_eval[key]
        fila["Parsimonious Original"] = f"{val:.1%}" if es_porcentaje else val
    else:
        fila["Parsimonious Original"] = "N/A"
        
    # Test
    if key in test_eval:
        val = test_eval[key]
        fila["Nueva Prueba (Test)"] = f"{val:.1%}" if es_porcentaje else val
    else:
        fila["Nueva Prueba (Test)"] = "—"
        
    filas_comparativa.append(fila)

agregar_fila("Número de Nodos", "gen_nodes", es_porcentaje=False)
agregar_fila("Número de Aristas", "gen_edges", es_porcentaje=False)
agregar_fila("Service F1 (Unique)", "svc_f1")
agregar_fila("Service Precision", "svc_precision")
agregar_fila("Service Recall", "svc_recall")
agregar_fila("Edge F1 (Connections)", "edge_f1")
agregar_fila("Edge Precision", "edge_precision")
agregar_fila("Edge Recall", "edge_recall")

df_resumen = pd.DataFrame(filas_comparativa)
print("📊 RESUMEN COMPARATIVO DE RENDIMIENTO:")
display(df_resumen)

# Detalles cualitativos de errores
print("\n🔍 ERRORES O DETALLES EN LA NUEVA PRUEBA (TEST):")
print(f"  • Servicios Faltantes (Omitidos):   {test_eval.get('services_missing', [])}")
print(f"  • Servicios Alucinados (Inventados): {test_eval.get('services_hallucinated', [])}")

if parsi_eval:
    print("\n🔄 CAMBIOS CONTRA PARSIMONIOUS ORIGINAL:")
    print(f"  • Faltaban antes: {parsi_eval.get('services_missing', [])}")
    print(f"  • Faltan ahora:   {test_eval.get('services_missing', [])}")
    print(f"  • Alucinaba antes: {parsi_eval.get('services_hallucinated', [])}")
    print(f"  • Alucina ahora:   {test_eval.get('services_hallucinated', [])}")

📊 RESUMEN COMPARATIVO DE RENDIMIENTO:


,Métrica,Ground Truth,Standard Original,Parsimonious Original,Nueva Prueba (Test)
0,Número de Nodos,9,11,11,11
1,Número de Aristas,8,4,10,4
2,Service F1 (Unique),—,85.7%,85.7%,85.7%
3,Service Precision,—,85.7%,85.7%,85.7%
4,Service Recall,—,85.7%,85.7%,85.7%
5,Edge F1 (Connections),—,66.7%,55.6%,66.7%
6,Edge Precision,—,100.0%,50.0%,100.0%
7,Edge Recall,—,50.0%,62.5%,50.0%



🔍 ERRORES O DETALLES EN LA NUEVA PRUEBA (TEST):
  • Servicios Faltantes (Omitidos):   ['UserCompanyDeveloper']
  • Servicios Alucinados (Inventados): ['UserCompanyAgent']

🔄 CAMBIOS CONTRA PARSIMONIOUS ORIGINAL:
  • Faltaban antes: ['UserCompanyDeveloper']
  • Faltan ahora:   ['UserCompanyDeveloper']
  • Alucinaba antes: ['UserCompanyAnalyst']
  • Alucina ahora:   ['UserCompanyAgent']


## Prompt version standard v2

In [39]:
import base64
import json
import csv
import re
from google import genai
from config.settings import GEMINI_API_KEY, GEMINI_MODEL

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# =====================================================================
# 1. CARGA DE RECURSOS
# =====================================================================
image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

with open(ws_transcript, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# Cargar catálogo de servicios y actores
services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

aws_services_str = ", ".join(sorted(aws_services_list))
user_actors_str = ", ".join(sorted(user_actors_list))

# =====================================================================
# 2. DEFINICIÓN DE PROMPTS CON ENFOQUE VISUAL (FASE 1 y FASE 2)
# =====================================================================
MFR_STAGE_1_PROMPT_TEMPLATE = """You are an expert cloud architecture ontology modeler.
Your task is to analyze the provided whiteboard diagram and audio transcript to build a structured "World Model" representing the architecture components and their physical drawn connections.

Do NOT generate the final graph, node IDs, or edge sequences. Focus strictly on identifying the entities and the VISUAL CONNECTIONS drawn on the whiteboard.

## ENTITY IDENTIFICATION RULES:
1. Identify all active systems, databases, cloud services, and actors.
2. Use valid AWS services from this list: <AWS_SERVICES_PLACEHOLDER>.
3. Identify actors/users from this list: <USER_ACTORS_PLACEHOLDER>. Map internal operations/migration teams to "UserCompanyAgent".
4. Map on-premises/external systems to "ThirdParty" (e.g. legacy databases, Slack, or on-premises servers), unless a physically drawn corporate data center box is shown.
5. Do NOT include transient files, packages, or disk images (like AMIs, zip files, or container images) as entities. They are transition properties, not permanent nodes.

## VISUAL CONNECTION RULES:
1. Look closely at the whiteboard image. Identify every drawn connection line or arrow between the entities.
2. Note the source, target, and the direction of the arrow.
3. Only list connections that are VISUALLY represented by lines or arrows on the board. Do NOT list connections that are only mentioned in the audio but have no line drawn on the board.

## FORMAT RULES:
- Inside JSON string fields, NEVER use unescaped double quotes. Use single quotes instead (e.g. write 'AMI' instead of "AMI").

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "entities": [
    {"service": "AWS Service or Actor Name", "name": "Label on the whiteboard", "type": "AWS_Service|User|ThirdParty", "rationale": "..."}
  ],
  "visual_connections": [
    {"source_label": "Label of source box", "target_label": "Label of target box", "arrow_direction": "source_label -> target_label", "description": "What this line represents"}
  ]
}
"""

MFR_STAGE_2_PROMPT_TEMPLATE = """You are an expert AWS Solutions Architect. Your task is to compile the final cloud architecture graph from the provided World Model, whiteboard image, and audio transcript.

You must follow the structure and level of detail of the Cloudscape dataset schema.

## WORLD MODEL (IMMUTABLE INPUT):
Below is the verified World Model of the architecture. You MUST strictly adhere to this model.
<WORLD_MODEL_PLACEHOLDER>

## RULES FOR NODES:
1. Create nodes ONLY for the services and actors listed in the "entities" section of the World Model. Do NOT invent or add any other services, actors, or transient nodes (such as AMIs, packages, or config files).

## RULES FOR EDGES:
1. Create edges ONLY for the connections listed in the "visual_connections" section of the World Model.
2. Keep the exact source and target direction as specified in the "visual_connections" (arrow_direction).
3. Do NOT create return/response paths or API acknowledgments (e.g., target sending back status) unless they are physically drawn as a separate arrow on the whiteboard.
4. Set "type" to "control" for system triggers, API calls, scripts, or command invocations. Set "type" to "data" ONLY for actual block data replication, payload transmission, or database reads/writes.
5. Set the "notes" field for each edge to describe the interaction, combining the transcript details with the visual path.
6. The `flow_id` represents unique workflows. Sequential steps in the same workflow must share the same `flow_id` and have sequential `seq` numbers (0, 1, 2, 3...) based on the transcript chronology.

## FORMAT RULES:
- Inside JSON string fields, NEVER use unescaped double quotes. Use single quotes instead (e.g. write 'AMI' instead of "AMI").

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "step_by_step_reasoning": "Deduce the nodes and edges from the World Model entities and visual connections, ordering them chronologically...",
  "graph": {
    "name": "<title of the architecture>",
    "link": "<youtube URL if known, else empty string>",
    "categories": "<comma-separated from: data_ingestion, interactive, compute_intensive, control, other>",
    "graph_usable": true,
    "notes": "<distilled context>"
  },
  "nodes": [
    {"id": "0", "service": "service name from entities", "name": "name from entities", "notes": "context of usage"}
  ],
  "edges": [
    {"source": "node_id", "target": "node_id", "flow_id": 0, "seq": "0", "type": "data|control", "notes": "action detail"}
  ]
}
"""

# Robust JSON text cleaner to prevent syntax errors
def clean_json_text(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        if "```" in text:
            text = text[:text.rfind("```")]
    text = text.strip()
    # Remove trailing commas in lists or objects
    text = re.sub(r',\s*([\]}])', r'\1', text)
    return text

# =====================================================================
# 3. FASE 1: CONSTRUCCIÓN DEL MODELO DEL MUNDO (MODELER AGENT - CON VISIÓN)
# =====================================================================
print("--- FASE 1: Construcción del Modelo del Mundo (Gemini Vision) ---")
formatted_prompt_1 = MFR_STAGE_1_PROMPT_TEMPLATE.replace(
    "<USER_ACTORS_PLACEHOLDER>", user_actors_str
).replace(
    "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
)
full_prompt_1 = f"{formatted_prompt_1}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

print(f"📡 Enviando imagen y transcripción al Agente Modelador ({GEMINI_MODEL})...")
response_1 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_1},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

world_model_text = clean_json_text(response_1.text)
world_model = json.loads(world_model_text)

# Guardar World Model para debug
ws_world_model_json = LAB_WORKSPACE / "world_model.json"
ws_world_model_json.write_text(json.dumps(world_model, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ Modelo del Mundo generado con éxito ({len(world_model.get('entities', []))} entidades, {len(world_model.get('visual_connections', []))} conexiones visuales).")
print(f"  Guardado en: {ws_world_model_json}")

# =====================================================================
# 4. FASE 2: PLANIFICACIÓN Y RAZONAMIENTO DEL GRAFO (PLANNER AGENT - CON VISIÓN)
# =====================================================================
print("\n--- FASE 2: Planificación y Generación del Grafo (Gemini Vision + Reasoning) ---")
formatted_prompt_2 = MFR_STAGE_2_PROMPT_TEMPLATE.replace(
    "<WORLD_MODEL_PLACEHOLDER>", json.dumps(world_model, indent=2, ensure_ascii=False)
)
full_prompt_2 = f"{formatted_prompt_2}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

print(f"📡 Enviando World Model, transcripción e imagen al Agente Planificador...")
response_2 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_2},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

graph_json_text = clean_json_text(response_2.text)
test_result = json.loads(graph_json_text)

# Guardar resultado del experimento localmente
ws_test_json = LAB_WORKSPACE / "test_analysis.json"
ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"✅ Análisis MFR (Visual 2-Fases) completado. Grafo guardado en: {ws_test_json}")


--- FASE 1: Construcción del Modelo del Mundo (Gemini Vision) ---
📡 Enviando imagen y transcripción al Agente Modelador (gemini-3.5-flash)...
✓ Modelo del Mundo generado con éxito (11 entidades, 10 conexiones visuales).
  Guardado en: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/Cgv0kfp_6xQ/world_model.json

--- FASE 2: Planificación y Generación del Grafo (Gemini Vision + Reasoning) ---
📡 Enviando World Model, transcripción e imagen al Agente Planificador...
✅ Análisis MFR (Visual 2-Fases) completado. Grafo guardado en: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/Cgv0kfp_6xQ/test_analysis.json


## Prompt Claude

In [3]:
"""
MFR v2 IMPROVED — Double-Model with fused Parsimonious rules
=============================================================
CAMBIOS vs el doble-modelo original:
  1. Stage 1: Agrega "PARSIMONY PRINCIPLE" y conteo de iconos físicos
  2. Stage 1: Agrega "NON-CLOUD NORMALIZATION" del parsimonious
  3. Stage 1: Agrega "USER ACTOR NORMALIZATION" detallada del parsimonious
  4. Stage 2: Edge type alineado al GT → "data" y "meta" (no "control")
  5. Stage 2: Convención explícita de dirección de aristas
  6. Stage 2: Anti-retorno reforzado con redacción mejorada
  7. Stage 2: Principio de parsimonia en aristas

Instrucciones:
  - Ejecutar las celdas 1 y 2 del notebook antes de esta celda
  - Las variables ws_whiteboard, ws_transcript, LAB_WORKSPACE, PROJECT_ROOT,
    VIDEO_ID ya deben estar definidas
"""

import base64
import json
import csv
import re
from google import genai
from config.settings import GEMINI_API_KEY, GEMINI_MODEL

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# =====================================================================
# 1. CARGA DE RECURSOS
# =====================================================================
image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

with open(ws_transcript, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# Cargar catálogo de servicios y actores
services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

aws_services_str = ", ".join(sorted(aws_services_list))
user_actors_str = ", ".join(sorted(user_actors_list))

# =====================================================================
# 2. DEFINICIÓN DE PROMPTS — MFR v2 IMPROVED
# =====================================================================

# ── STAGE 1: MODELER — Extraer World Model visual ──────────────────
MFR_STAGE_1_PROMPT_TEMPLATE = """You are an expert cloud architecture ontology modeler.
Your task is to analyze the provided whiteboard diagram and audio transcript to build a structured "World Model" representing the architecture components and their physical drawn connections.

Do NOT generate the final graph, node IDs, or edge sequences. Focus strictly on identifying the entities and the VISUAL CONNECTIONS drawn on the whiteboard.

## ENTITY IDENTIFICATION RULES:
1. Identify all active systems, databases, cloud services, and actors VISIBLE as physical icons (boxes, circles, or labeled regions) on the whiteboard.
2. Use valid AWS services from this list: <AWS_SERVICES_PLACEHOLDER>.
3. USER ACTOR NORMALIZATION: Only add User nodes that are EXPLICITLY shown as icons on the whiteboard OR mentioned as main actors in the transcript. Choose from this list: <USER_ACTORS_PLACEHOLDER>. To match Ground Truth style:
   - Map end-users accessing via browsers to "UserConsumerWeb", and app users to "UserConsumerMobile". Do NOT combine them into "UserConsumerWebMobile" unless a single physical box on the board is explicitly labeled for both.
   - Prefer "UserCompanyAgent" for internal operations teams, database administrators, migration teams, or backend system operators.
   - Use "UserCompanyDeveloper" ONLY when the text or diagram explicitly refers to writing application code, managing CI/CD pipelines, or software development.
4. NON-CLOUD & ON-PREMISE NORMALIZATION: Do NOT use "ThirdParty" for internal microservices. Map them to the underlying AWS compute/storage service they run on (e.g. "EKS", "Lambda").
   - However, map on-premises servers, local databases, and legacy infrastructure to "ThirdParty" (representing external resources outside AWS), unless a dedicated data center icon is explicitly drawn (in which case use "OnPremDC").
5. Do NOT include transient files, packages, or disk images (like AMIs, zip files, or container images) as entities. They are transition properties, not permanent nodes.

## NODE COUNT RULE (CRITICAL):
The number of entities you identify must match the number of PHYSICAL ICONS (boxes, circles, labeled regions) drawn on the whiteboard. Do NOT add entities that are only mentioned verbally but have no physical icon on the board. Do NOT duplicate entities to represent logical sub-components unless they have distinct physical icons.

## VISUAL CONNECTION RULES:
1. Look closely at the whiteboard image. Identify every drawn connection line or arrow between the entities.
2. Note the source, target, and the direction of the arrow.
3. Only list connections that are VISUALLY represented by lines or arrows on the board. Do NOT list connections that are only mentioned in the audio but have no line drawn on the board.

## PARSIMONY PRINCIPLE:
Prefer FEWER entities and connections over more. If you are unsure whether a service or connection exists on the whiteboard, DO NOT include it. It is better to miss a real entity than to hallucinate a fake one.

## FORMAT RULES:
- Inside JSON string fields, NEVER use unescaped double quotes. Use single quotes instead (e.g. write 'AMI' instead of "AMI").

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "entities": [
    {"service": "AWS Service or Actor Name", "name": "Label on the whiteboard", "type": "AWS_Service|User|ThirdParty", "rationale": "..."}
  ],
  "visual_connections": [
    {"source_label": "Label of source box", "target_label": "Label of target box", "arrow_direction": "source_label -> target_label", "description": "What this line represents"}
  ]
}
"""

# ── STAGE 2: PLANNER — Compilar grafo final ────────────────────────
MFR_STAGE_2_PROMPT_TEMPLATE = """You are an expert AWS Solutions Architect. Your task is to compile the final cloud architecture graph from the provided World Model, whiteboard image, and audio transcript.

You must follow the structure and level of detail of the Cloudscape dataset schema (FAST25 paper by Satija et al.).

## WORLD MODEL (IMMUTABLE INPUT):
Below is the verified World Model of the architecture. You MUST strictly adhere to this model.
<WORLD_MODEL_PLACEHOLDER>

## RULES FOR NODES:
1. Create nodes ONLY for the services and actors listed in the "entities" section of the World Model. Do NOT invent or add any other services, actors, or transient nodes (such as AMIs, packages, or config files).

## RULES FOR EDGES:
1. Create edges ONLY for the connections listed in the "visual_connections" section of the World Model.
2. Keep the exact source and target direction as specified in the "visual_connections" (arrow_direction).
3. EDGE DIRECTIONALITY (NO RETURN PATHS): Map ONLY active data movement or control triggers. Do NOT add return/response paths or API acknowledgments (e.g., target sending back status) unless they are physically drawn as a SEPARATE, DISTINCT arrow on the whiteboard. Orient arrows in the direction of request initiation.
4. EDGE DIRECTION CONVENTION:
   - Always follow the direction of DATA FLOW (who sends data to whom).
   - If Service A reads from Service B, the edge is B -> A (B sends data to A).
   - If Service A triggers/invokes Service B, the edge is A -> B.
   - For bidirectional data exchange, model ONLY the primary/initiating direction unless the World Model lists both directions as separate visual connections.
5. Edge type: Set "type" to "data" for all edges by default. Only use "meta" for edges that represent monitoring/observability signals (e.g., to CloudWatch) or orchestration triggers (e.g., from StepFunctions).
6. Set the "notes" field for each edge to describe the interaction, combining the transcript details with the visual path.
7. The `flow_id` represents unique workflows. Sequential steps in the same workflow must share the same `flow_id` and have sequential `seq` numbers (0, 1, 2, 3...) based on the transcript chronology.
8. Minimize the number of flows. Group related sequential interactions into a single flow.

## PARSIMONY PRINCIPLE FOR EDGES:
Prefer FEWER edges over more. If a connection is ambiguous or unclear in the World Model, DO NOT create an edge for it. It is better to miss a real edge than to invent a wrong one.

## FORMAT RULES:
- Inside JSON string fields, NEVER use unescaped double quotes. Use single quotes instead (e.g. write 'AMI' instead of "AMI").

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "step_by_step_reasoning": "Deduce the nodes and edges from the World Model entities and visual connections, ordering them chronologically. Apply the edge direction convention...",
  "graph": {
    "name": "<title of the architecture>",
    "link": "<youtube URL if known, else empty string>",
    "categories": "<comma-separated from: data_ingestion, interactive, compute_intensive, control, other>",
    "graph_usable": true,
    "notes": "<distilled context>"
  },
  "nodes": [
    {"id": "0", "service": "service name from entities", "name": "name from entities", "notes": "context of usage"}
  ],
  "edges": [
    {"source": "node_id", "target": "node_id", "flow_id": 0, "seq": "0", "type": "data", "notes": "action detail"}
  ]
}
"""

# Robust JSON text cleaner to prevent syntax errors
def clean_json_text(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        if "```" in text:
            text = text[:text.rfind("```")]
    text = text.strip()
    # Remove trailing commas in lists or objects
    text = re.sub(r',\s*([\]}])', r'\1', text)
    return text

# =====================================================================
# 3. FASE 1: CONSTRUCCIÓN DEL MODELO DEL MUNDO (MODELER AGENT)
# =====================================================================
print("=" * 60)
print("  FASE 1: Construcción del Modelo del Mundo (Gemini Vision)")
print("=" * 60)
formatted_prompt_1 = MFR_STAGE_1_PROMPT_TEMPLATE.replace(
    "<USER_ACTORS_PLACEHOLDER>", user_actors_str
).replace(
    "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
)
full_prompt_1 = f"{formatted_prompt_1}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

print(f"📡 Enviando imagen y transcripción al Agente Modelador ({GEMINI_MODEL})...")
response_1 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_1},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

world_model_text = clean_json_text(response_1.text)
world_model = json.loads(world_model_text)

# Guardar World Model para debug
ws_world_model_json = LAB_WORKSPACE / "world_model_v2.json"
ws_world_model_json.write_text(json.dumps(world_model, indent=2, ensure_ascii=False), encoding="utf-8")

n_entities = len(world_model.get('entities', []))
n_connections = len(world_model.get('visual_connections', []))
print(f"✓ Modelo del Mundo generado: {n_entities} entidades, {n_connections} conexiones visuales")
print(f"  Guardado en: {ws_world_model_json}")

# Mostrar resumen del World Model
print("\n📋 Entidades detectadas:")
for ent in world_model.get("entities", []):
    print(f"  • [{ent.get('type', '?')}] {ent.get('service', '?')} — {ent.get('name', '?')}")
print(f"\n🔗 Conexiones visuales detectadas:")
for conn in world_model.get("visual_connections", []):
    print(f"  • {conn.get('arrow_direction', '?')} — {conn.get('description', '?')}")

# =====================================================================
# 4. FASE 2: PLANIFICACIÓN Y GENERACIÓN DEL GRAFO (PLANNER AGENT)
# =====================================================================
print("\n" + "=" * 60)
print("  FASE 2: Planificación y Generación del Grafo (Planner)")
print("=" * 60)
formatted_prompt_2 = MFR_STAGE_2_PROMPT_TEMPLATE.replace(
    "<WORLD_MODEL_PLACEHOLDER>", json.dumps(world_model, indent=2, ensure_ascii=False)
)
full_prompt_2 = f"{formatted_prompt_2}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

print(f"📡 Enviando World Model, transcripción e imagen al Agente Planificador ({GEMINI_MODEL})...")
response_2 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_2},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

graph_json_text = clean_json_text(response_2.text)
test_result = json.loads(graph_json_text)

# Guardar resultado del experimento localmente
ws_test_json = LAB_WORKSPACE / "test_analysis.json"
ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")

n_nodes = len(test_result.get("nodes", []))
n_edges = len(test_result.get("edges", []))
print(f"\n✅ Grafo MFR v2 generado: {n_nodes} nodos, {n_edges} aristas")
print(f"   Guardado en: {ws_test_json}")

# Mostrar resumen del grafo
print("\n📋 Nodos generados:")
for node in test_result.get("nodes", []):
    print(f"  • [{node.get('id')}] {node.get('service', '?')} — {node.get('name', '?')}")
print(f"\n🔗 Aristas generadas:")
for edge in test_result.get("edges", []):
    print(f"  • {edge.get('source')} → {edge.get('target')} [{edge.get('type', 'data')}] flow={edge.get('flow_id')}")


  FASE 1: Construcción del Modelo del Mundo (Gemini Vision)
📡 Enviando imagen y transcripción al Agente Modelador (gemini-3.5-flash)...
✓ Modelo del Mundo generado: 11 entidades, 4 conexiones visuales
  Guardado en: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/-kA0ahrhX3I/world_model_v2.json

📋 Entidades detectadas:
  • [User] UserCompanyAgent — Security Engineers
  • [AWS_Service] Athena — Amazon Athena
  • [AWS_Service] QuickSight — Amazon QuickSight
  • [ThirdParty] ThirdParty — MySQL
  • [AWS_Service] Glue — AWS Glue
  • [AWS_Service] S3 — S3 Bucket (Findings)
  • [AWS_Service] Macie — Amazon Macie (Center)
  • [AWS_Service] Macie — Amazon Macie (Top Right)
  • [AWS_Service] S3 — Amazon Simple Storage Service (Amazon S3) (Top Right)
  • [AWS_Service] Macie — Amazon Macie (Bottom Right)
  • [AWS_Service] S3 — Amazon Simple Storage Service (Amazon S3) (Bottom Right)

🔗 Conexiones visuales detectadas:
  • Amazon Macie (Center) -> S3 Bucket (Findings)